# Data Cleaning -- Test Dataset
**This notebook prepares Flatiron Health CSV files for patients with advanced non-small cell lung cancer in anticipation of training a gradient-boosted survival model to predict mortality from time of first-line treatment. Specifically, it processes and cleans the test cohort.**

## Import packages

In [1]:
import numpy as np
import pandas as pd

from flatiron_cleaner import DataProcessorNSCLC, merge_dataframes

## Clean CSV files 

In [2]:
test_ids = pd.read_csv('../outputs/test_patient_ids.csv')

In [3]:
test_ids = test_ids.PatientID.to_list()

In [4]:
index_date_df = pd.read_csv('../data/LineOfTherapy.csv')

In [5]:
index_date_df = (
    index_date_df
    .query('LineNumber == 1')
    .query('IsMaintenanceTherapy == False')
    [['PatientID', 'StartDate']]
)

In [6]:
index_date_df.shape

(83095, 2)

In [7]:
index_date_df = (
    index_date_df[index_date_df.PatientID.isin(test_ids)]
    [['PatientID', 'StartDate']]
)

In [8]:
index_date_df.shape

(16619, 2)

In [9]:
processor = DataProcessorNSCLC()

### Process Enhanced_AdvancedNSCLC.csv

In [10]:
enhanced_df = processor.process_enhanced(file_path = '../data/Enhanced_AdvancedNSCLC.csv',
                                         patient_ids = test_ids,
                                         drop_dates = False)

2026-03-16 14:14:48,785 - INFO - Successfully read Enhanced_AdvancedNSCLC.csv file with shape: (114756, 6) and unique PatientIDs: 114756
2026-03-16 14:14:48,785 - INFO - Filtering for 16619 specific PatientIDs
2026-03-16 14:14:48,795 - INFO - Successfully filtered Enhanced_AdvancedNSCLC.csv file with shape: (16619, 6) and unique PatientIDs: 16619
2026-03-16 14:14:48,810 - INFO - Successfully processed Enhanced_AdvancedNSCLC.csv file with final shape: (16619, 8) and unique PatientIDs: 16619


In [11]:
index_date_df['StartDate'] = pd.to_datetime(index_date_df['StartDate'])

enhanced_df = pd.merge(enhanced_df, index_date_df[['PatientID', 'StartDate']], on = 'PatientID', how = 'left')
enhanced_df['days_adv_to_treatment'] = (enhanced_df['StartDate'] - enhanced_df['AdvancedDiagnosisDate']).dt.days
enhanced_df['days_adv_to_treatment'] = np.where(enhanced_df['days_adv_to_treatment'] < 0 , 0, enhanced_df['days_adv_to_treatment'])

In [12]:
enhanced_df['days_diagnosis_to_adv'] = np.where(enhanced_df['days_diagnosis_to_adv'] < 0 , 0, enhanced_df['days_diagnosis_to_adv'])
enhanced_df['days_diagnosis_to_adv'] = enhanced_df['days_diagnosis_to_adv'].fillna(0)

In [13]:
enhanced_df['GroupStage_mod'] = enhanced_df["GroupStage_mod"].map({
    '0': 0, 
    'I': 1,
    'II': 2,
    'III': 3,
    'IV': 4,
    'unknown': np.nan
})

enhanced_df['GroupStage_mod_na'] = np.where(enhanced_df['GroupStage_mod'].isna(), 1, 0)

# impute 4 since stage IV given most common
enhanced_df['GroupStage_mod'] = enhanced_df['GroupStage_mod'].fillna(4)

In [14]:
enhanced_df['Histology'] = enhanced_df['Histology'].fillna('Non-squamous cell carcinoma')

In [15]:
enhanced_df = enhanced_df.drop(columns = ['DiagnosisDate', 
                                          'AdvancedDiagnosisDate', 
                                          'StartDate'])

### Process Demographics.csv 

In [16]:
demographics_df = processor.process_demographics(file_path = '../data/Demographics.csv',
                                                 index_date_df = index_date_df,
                                                 index_date_column = 'StartDate')

2026-03-16 14:14:48,903 - INFO - Successfully read Demographics.csv file with shape: (114756, 6) and unique PatientIDs: 114756
2026-03-16 14:14:48,938 - INFO - Successfully processed Demographics.csv file with final shape: (16619, 6) and unique PatientIDs: 16619


In [17]:
# Impute missing with male
demographics_df['sex_male'] = np.where(demographics_df['Gender'] == 'F', 0, 1)

In [18]:
demographics_df = demographics_df.drop(columns = ['Gender'])

### Process Enhanced_AdvNSCLCBiomarkers.csv

In [19]:
biomarkers_df = processor.process_biomarkers(file_path = '../data/Enhanced_AdvNSCLCBiomarkers.csv',
                                             index_date_df = index_date_df, 
                                             index_date_column = 'StartDate',
                                             days_before = None, 
                                             days_after = 30)

2026-03-16 14:14:50,002 - INFO - Successfully read Enhanced_AdvNSCLCBiomarkers.csv file with shape: (1021678, 18) and unique PatientIDs: 93715
2026-03-16 14:14:50,266 - INFO - Successfully merged Enhanced_AdvNSCLCBiomarkers.csv df with index_date_df resulting in shape: (168696, 19) and unique PatientIDs: 14391
2026-03-16 14:14:50,993 - INFO - Successfully processed Enhanced_AdvNSCLCBiomarkers.csv file with final shape: (16619, 11) and unique PatientIDs: 16619


In [20]:
biomarkers_df['PDL1_percent_staining'] = biomarkers_df["PDL1_percent_staining"].map({
    '0%': '0%', 
    '<1%': '1-49%',
    '1%': '1-49%',
    '2% - 4%': '1-49%',
    '5% - 9%': '1-49%', 
    '10% - 19%': '1-49%',
    '20% - 29%': '1-49%',
    '30% - 39%': '1-49%',
    '40% - 49%': '1-49%',
    '50% - 59%': '>=50%', 
    '60% - 69%': '>=50%', 
    '70% - 79%': '>=50%',
    '80% - 89%': '>=50%', 
    '90% - 99%': '>=50%', 
    '100%': '>=50%',
})

biomarkers_df['PDL1_percent_staining'] = biomarkers_df['PDL1_percent_staining'].fillna('unknown')
biomarkers_df['PDL1_percent_staining'] = biomarkers_df['PDL1_percent_staining'].astype('category')

In [21]:
for biomarker in biomarkers_df.columns.drop('PDL1_percent_staining'):
    biomarkers_df[biomarker] = biomarkers_df[biomarker].fillna('unknown')
    biomarkers_df[biomarker].value_counts(dropna = False)

### Process ECOG.csv

In [22]:
ecog_df = processor.process_ecog(file_path = '../data/ECOG.csv', 
                                 index_date_df = index_date_df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 days_before_further = 180)

2026-03-16 14:14:51,546 - INFO - Successfully read ECOG.csv file with shape: (1623197, 4) and unique PatientIDs: 85577
2026-03-16 14:14:51,780 - INFO - Successfully merged ECOG.csv df with index_date_df resulting in shape: (304494, 5) and unique PatientIDs: 13346
2026-03-16 14:14:51,976 - INFO - Successfully processed ECOG.csv file with final shape: (16619, 3) and unique PatientIDs: 16619


In [23]:
ecog_df['ecog_index'] = ecog_df['ecog_index'].astype('float64')
ecog_df['ecog_index_na'] = np.where(ecog_df['ecog_index'].isna(), 1, 0)

# impute 1 for missing ECOG since most common
ecog_df['ecog_index'] = ecog_df['ecog_index'].fillna(1)

In [24]:
ecog_df['ecog_newly_gte2'] = ecog_df['ecog_newly_gte2'].fillna(0)

### Process Vitals.csv

In [25]:
vitals_df = processor.process_vitals(file_path = '../data/Vitals.csv',
                                     index_date_df = index_date_df,
                                     index_date_column = 'StartDate',
                                     weight_days_before = 90,
                                     days_after = 0,
                                     vital_summary_lookback = 180, 
                                     abnormal_reading_threshold = 1)

2026-03-16 14:16:01,236 - INFO - Successfully read Vitals.csv file with shape: (29862758, 16) and unique PatientIDs: 114285
2026-03-16 14:16:11,279 - INFO - Successfully merged Vitals.csv df with index_date_df resulting in shape: (5206450, 17) and unique PatientIDs: 16615
2026-03-16 14:16:13,582 - INFO - Successfully processed Vitals.csv file with final shape: (16619, 8) and unique PatientIDs: 16619


### Process Lab.csv

In [26]:
labs_df = processor.process_labs(file_path = '../data/Lab.csv',
                                 index_date_df = index_date_df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 summary_lookback = 180)

2026-03-16 14:18:07,058 - INFO - Successfully read Lab.csv file with shape: (82093498, 17) and unique PatientIDs: 108761
2026-03-16 14:18:28,293 - INFO - Successfully merged Lab.csv df with index_date_df resulting in shape: (14691788, 18) and unique PatientIDs: 16411
2026-03-16 14:18:58,079 - INFO - Successfully processed Lab.csv file with final shape: (16619, 76) and unique PatientIDs: 16619


### Process MedicationAdministration.csv

In [27]:
medications_df = processor.process_medications(file_path = '../data/MedicationAdministration.csv',
                                               index_date_df = index_date_df,
                                               index_date_column = 'StartDate',
                                               days_before = 90,
                                               days_after = 0)

2026-03-16 14:19:06,906 - INFO - Successfully read MedicationAdministration.csv file with shape: (7486734, 11) and unique PatientIDs: 89605
2026-03-16 14:19:08,632 - INFO - Successfully merged MedicationAdministration.csv df with index_date_df resulting in shape: (1313722, 12) and unique PatientIDs: 15412
2026-03-16 14:19:08,736 - INFO - Successfully processed MedicationAdministration.csv file with final shape: (16619, 9) and unique PatientIDs: 16619


### Process Diagnosis.csv

In [28]:
diagnosis_df = processor.process_diagnosis(file_path = '../data/Diagnosis.csv',
                                           index_date_df = index_date_df,
                                           index_date_column = 'StartDate',
                                           days_before = None,
                                           days_after = 0)

2026-03-16 14:19:14,247 - INFO - Successfully read Diagnosis.csv file with shape: (9083016, 6) and unique PatientIDs: 114756
2026-03-16 14:19:15,271 - INFO - Successfully merged Diagnosis.csv df with index_date_df resulting in shape: (1459083, 7) and unique PatientIDs: 16619
2026-03-16 14:19:18,340 - INFO - Successfully processed Diagnosis.csv file with final shape: (16619, 39) and unique PatientIDs: 16619


### Process Enhanced_Mortality_V2.csv

In [29]:
mortality_df = processor.process_mortality(file_path = '../data/Enhanced_Mortality_V2.csv',
                                           index_date_df = index_date_df, 
                                           index_date_column = 'StartDate',
                                           visit_path = '../data/Visit.csv', 
                                           telemedicine_path = '../data/Telemedicine.csv', 
                                           biomarkers_path = '../data/Enhanced_AdvNSCLCBiomarkers.csv', 
                                           oral_path = '../data/Enhanced_AdvNSCLC_Orals.csv',
                                           progression_path = '../data/Enhanced_AdvNSCLC_Progression.csv',
                                           drop_dates = False)

2026-03-16 14:19:18,410 - INFO - Successfully read Enhanced_Mortality_V2.csv file with shape: (84881, 2) and unique PatientIDs: 84881
2026-03-16 14:19:18,461 - INFO - Successfully merged Enhanced_Mortality_V2.csv df with index_date_df resulting in shape: (16619, 3) and unique PatientIDs: 16619
2026-03-16 14:19:22,385 - INFO - The following columns ['last_visit_date', 'last_biomarker_date', 'last_oral_date', 'last_progression_date'] are used to calculate the last EHR date
2026-03-16 14:19:22,395 - INFO - Successfully processed Enhanced_Mortality_V2.csv file with final shape: (16619, 6) and unique PatientIDs: 16619. There are 0 out of 16619 patients with missing duration values


In [30]:
mortality_df = mortality_df[['PatientID', 'event', 'duration']]

In [31]:
mortality_df = mortality_df.query('duration >= 0')

## Merge dataframes

In [32]:
testing_df = merge_dataframes(enhanced_df,
                              demographics_df,
                              biomarkers_df,
                              ecog_df,
                              vitals_df,
                              labs_df,
                              medications_df,
                              diagnosis_df, 
                              mortality_df,
                              merge_type = 'inner')

2026-03-16 14:19:22,522 - INFO - Anticipated number of merges: 8
2026-03-16 14:19:22,523 - INFO - Anticipated number of columns in final dataframe presuming all columns are unique except for PatientID: 156
2026-03-16 14:19:22,525 - INFO - Dataset 1 shape: (16619, 8), unique PatientIDs: 16619
2026-03-16 14:19:22,527 - INFO - Dataset 2 shape: (16619, 6), unique PatientIDs: 16619
2026-03-16 14:19:22,529 - INFO - Dataset 3 shape: (16619, 11), unique PatientIDs: 16619
2026-03-16 14:19:22,531 - INFO - Dataset 4 shape: (16619, 4), unique PatientIDs: 16619
2026-03-16 14:19:22,533 - INFO - Dataset 5 shape: (16619, 8), unique PatientIDs: 16619
2026-03-16 14:19:22,534 - INFO - Dataset 6 shape: (16619, 76), unique PatientIDs: 16619
2026-03-16 14:19:22,536 - INFO - Dataset 7 shape: (16619, 9), unique PatientIDs: 16619
2026-03-16 14:19:22,537 - INFO - Dataset 8 shape: (16619, 39), unique PatientIDs: 16619
2026-03-16 14:19:22,540 - INFO - Dataset 9 shape: (16552, 3), unique PatientIDs: 16552
2026-03-

In [33]:
testing_df.shape

(16552, 156)

## Export dataframe

In [34]:
testing_df.to_csv('../outputs/1L_features_testing.csv', index = False)

In [35]:
# Save dtypes
testing_df.dtypes.apply(lambda x: x.name).to_csv('../outputs/1L_features_testing_dtypes.csv')